# SPICE → GDS Layout Generator
SSCS Chipathon 2026 — gLayout Track

Converts SPICE subcircuit netlist to DRC-clean GDSII layout.
Includes AC simulation, DRC/LVS/PEX verification.

**PDK**: gf180mcuD  |  **Supply**: 1.8 V

In [ ]:
import sys, os
from pathlib import Path
root = Path("/foss/designs/chipathon2026-D")
sys.path.insert(0, str(root))

from glayout import gf180
from gdsfactory import Component
import gdstk, IPython.display

from core import (
    spice_to_gds, display_component,
    run_ota_ac, run_drc, run_lvs, run_pex,
    compare_pre_post,
)

In [ ]:
# ==========================================
# 1. Define SPICE Netlist
# ==========================================
# Edit this cell to change your circuit.
# Models: nfet_03v3 (NMOS), pfet_03v3 (PMOS)
# Supply: 1.8V, body connections: NMOS→vss, PMOS→vdd

netlist = """
.lib "/foss/pdks/gf180mcuD/libs.tech/ngspice/sm141064.ngspice" typical
.subckt ota_simple vin_p vin_n vout vbias vdd vss
M1 n1 vin_p ntail vss nfet_03v3 W=10u L=1u
M2 vout vin_n ntail vss nfet_03v3 W=10u L=1u
M3 n1 n1 vdd vdd pfet_03v3 W=20u L=1u
M4 vout n1 vdd vdd pfet_03v3 W=20u L=1u
M5 ntail vbias vss vss nfet_03v3 W=15u L=1u
.ends
"""

In [ ]:
# ==========================================
# 2. Generate GDS Layout
# ==========================================

GDS_PATH = os.path.join(os.getcwd(), "out.gds")

result = spice_to_gds(netlist, mode="analog", add_labels=True)
result.write_gds(GDS_PATH)
print(f"Layout written to {GDS_PATH}")
display_component(result, scale=2)

In [ ]:
# ==========================================
# 3. Pre-Simulation (AC Analysis)
# ==========================================

pre = run_ota_ac(netlist, "ota_simple",
                  vdd=1.8, vcm=0.9, vbias=0.65, cload=2e-12)

print(f"DC Gain  = {pre['dc_gain_db']:.1f} dB")
print(f"GBW      = {pre['gbw_hz']/1e6:.1f} MHz")
print(f"Phase M. = {pre['phase_margin_deg']:.1f} deg")
print(f"f-3dB    = {pre['f_3db_hz']/1e3:.1f} kHz")
print(f"AC points = {len(pre['ac_data'])}")

In [ ]:
# ==========================================
# 4. DRC / LVS / PEX (optional)
# ==========================================
# Requires Magic, netgen, and PDK installed.

os.environ.setdefault("PDK_ROOT", "/foss/pdks")
os.environ.setdefault("PDK", "gf180mcuD")
os.environ.setdefault("PDKPATH", "/foss/pdks/gf180mcuD")
os.environ.setdefault("STD_CELL_LIBRARY", "gf180mcu_fd_sc_mcu7t5v0")

# Run layout with verification (generates DRC+LVS+PEX automatically):
# result = spice_to_gds(netlist, mode="analog", add_labels=True, run_checks=True)

# Or run individually:
# drc = run_drc("ota_simple.gds", cell_name="ota_simple")
# lvs = run_lvs("ota_simple.gds", netlist_content=netlist, cell_name="ota_simple")
# pex = run_pex("ota_simple.gds", cell_name="ota_simple", mode=2)
#
# # Compare pre-layout vs post-layout (PEX):
# cmp = compare_pre_post(netlist, pex["pex_path"], "ota_simple",
#                         vdd=1.8, vcm=0.9, vbias=0.65)
# print(f"PRE: {cmp['pre']['dc_gain_db']:.1f} dB")
# print(f"POST: {cmp['post']['dc_gain_db']:.1f} dB")